# Normalization: rescale and standardize

Two common preprocessing steps before combining rasters or feeding them into models:

- **rescale** maps values to a target range (default [0, 1]) using min-max normalization.
- **standardize** centers values at zero with unit variance (z-score normalization).

Both functions handle NaN and infinite values (they pass through unchanged) and work on all four xarray-spatial backends: NumPy, CuPy, Dask+NumPy, and Dask+CuPy.

In [ ]:
%matplotlib inline
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial.normalize import rescale, standardize
from xrspatial import generate_terrain

## Synthetic terrain

Generate a 500x500 elevation raster with values roughly in the 0-1200 range. We'll sprinkle in a few NaN cells to show how they're preserved.

In [ ]:
terrain = generate_terrain(canvas=xr.DataArray(np.zeros((500, 500)), dims=['y', 'x']))

# Add some NaN holes
rng = np.random.default_rng(42)
mask = rng.random(terrain.shape) < 0.005
terrain.values[mask] = np.nan

print(f"Shape: {terrain.shape}")
print(f"Range: {float(np.nanmin(terrain)):.1f} to {float(np.nanmax(terrain)):.1f}")
print(f"NaN cells: {int(np.isnan(terrain.values).sum())}")

fig, ax = plt.subplots(figsize=(7, 6))
terrain.plot.imshow(ax=ax, cmap='terrain', add_colorbar=True,
                    cbar_kwargs={'label': 'Elevation'})
ax.set_title('Raw elevation')
ax.set_axis_off()
plt.tight_layout()

## rescale: min-max normalization

By default, `rescale()` maps finite values to [0, 1]. You can supply a custom range with `new_min` and `new_max`.

In [ ]:
scaled_01 = rescale(terrain)
scaled_byte = rescale(terrain, new_min=0, new_max=255)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

terrain.plot.imshow(ax=axes[0], cmap='terrain', add_colorbar=True)
axes[0].set_title('Original')
axes[0].set_axis_off()

scaled_01.plot.imshow(ax=axes[1], cmap='terrain', add_colorbar=True)
axes[1].set_title('rescale() -> [0, 1]')
axes[1].set_axis_off()

scaled_byte.plot.imshow(ax=axes[2], cmap='terrain', add_colorbar=True)
axes[2].set_title('rescale(0, 255)')
axes[2].set_axis_off()

plt.tight_layout()

print(f"[0,1] range:   {float(np.nanmin(scaled_01)):.4f} to {float(np.nanmax(scaled_01)):.4f}")
print(f"[0,255] range: {float(np.nanmin(scaled_byte)):.1f} to {float(np.nanmax(scaled_byte)):.1f}")
print(f"NaN preserved: {int(np.isnan(scaled_01.values).sum())} cells")

## standardize: z-score normalization

`standardize()` subtracts the mean and divides by the standard deviation of finite values. The result has mean ~0 and std ~1. Use `ddof=1` for sample standard deviation.

In [ ]:
zscored = standardize(terrain)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

terrain.plot.imshow(ax=axes[0], cmap='terrain', add_colorbar=True)
axes[0].set_title('Original')
axes[0].set_axis_off()

zscored.plot.imshow(ax=axes[1], cmap='RdBu_r', add_colorbar=True,
                    cbar_kwargs={'label': 'Z-score'})
axes[1].set_title('standardize()')
axes[1].set_axis_off()

plt.tight_layout()

finite = zscored.values[np.isfinite(zscored.values)]
print(f"Mean:  {finite.mean():.2e}")
print(f"Std:   {finite.std():.6f}")
print(f"Range: {finite.min():.3f} to {finite.max():.3f}")

## Practical use case: combining layers with different scales

When combining elevation and slope into a composite index, the raw values live on different scales. Rescaling both to [0, 1] puts them on equal footing.

In [ ]:
from xrspatial import slope

slp = slope(terrain)

# Raw values are on very different scales
print(f"Elevation range: {float(np.nanmin(terrain)):.0f} to {float(np.nanmax(terrain)):.0f}")
print(f"Slope range:     {float(np.nanmin(slp)):.1f} to {float(np.nanmax(slp)):.1f}")

# Rescale both to [0, 1] and combine
elev_norm = rescale(terrain)
slope_norm = rescale(slp)

# Simple composite: high elevation + steep slope = high risk
composite = 0.6 * elev_norm + 0.4 * slope_norm

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

elev_norm.plot.imshow(ax=axes[0], cmap='terrain', add_colorbar=True)
axes[0].set_title('Elevation [0, 1]')
axes[0].set_axis_off()

slope_norm.plot.imshow(ax=axes[1], cmap='YlOrRd', add_colorbar=True)
axes[1].set_title('Slope [0, 1]')
axes[1].set_axis_off()

composite.plot.imshow(ax=axes[2], cmap='inferno', add_colorbar=True)
axes[2].set_title('Weighted composite (0.6 elev + 0.4 slope)')
axes[2].set_axis_off()

plt.tight_layout()

## Accessor syntax

Both functions are available through the `.xrs` accessor on DataArrays.

```python
import xrspatial

terrain.xrs.rescale()
terrain.xrs.standardize(ddof=1)
```

In [ ]:
import xrspatial  # registers .xrs accessor

accessor_result = terrain.xrs.rescale()
np.testing.assert_array_equal(accessor_result.values, scaled_01.values)
print("Accessor output matches function output.")